In [ ]:
!pip install transformers torch
import sys
sys.path.append('/content/korean-chatbot/src')

In [ ]:
from tokenizer import KoreanTokenizer
from model import Transformer

kt = KoreanTokenizer()
kt.load("tokenizer/tokenizer.json")

model = Transformer(vocab_size=kt.vocab_size)
print(f"vocab size: {kt.vocab_size}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_len=512):
        self.samples = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                ids = tokenizer.encode(line.strip())
                if len(ids) > 1:
                    ids = ids[:max_len]
                    self.samples.append(torch.tensor(ids))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_fn(batch):
    return torch.nn.utils.rnn.pad_sequence(batch, batch_first=True, padding_value=0)

dataset = TextDataset("data/train.txt", kt)
loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
print(f"배치 수: {len(loader)}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(3):
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        loss = model.loss(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} | loss: {total_loss/len(loader):.4f}")

In [ ]:
import os
os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/model.pt")
print("모델 저장 완료!")